# T05 — Synthetic Test Data

T04 evaluated the pipeline on 7 hand-written queries. That covers the known failure modes but not the breadth of the corpus. Writing hundreds of questions by hand is slow, so we let an LLM generate `(question, reference answer)` pairs from the indexed chunks.

1. **Hand-written samples**: few, targeted at known failure modes, checked against the documents
2. **Synthetic samples**: an LLM writes one question and answer per sampled chunk
3. **Export**: both sets in RAGAS format (`user_input`, `reference`)

## Setup

Requires `OPENAI_API_KEY` (locally: in `.env.local` at the repo root). Run T01 first: it builds the vector store the later notebooks reuse.

In [ ]:
import json
import random
import sys

from loguru import logger

from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from ki_kompass_rag.baseline_rag import DB_DIR, build_llm, load_or_build_vector_store
from ki_kompass_rag.evaluation import EVALUATION_QUERIES

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, _ = await load_or_build_vector_store()
llm = build_llm("openai")

## 1. Hand-Written Samples

Hand-written samples encode business intent: they test whether the system still invents the Lara pallet or treats a supplier claim as verified. Each needs the exact question, the category it tests, and a short correct answer that a reviewer has checked against the documents.

The failure-mode queries come from `EVALUATION_QUERIES` (the T04 test set); a few direct fact lookups are added here.

In [ ]:
manual_samples = [
    {
        "user_input": q["query"],
        "reference": q["ground_truth_answer"],
        "category": q["category"],
    }
    for q in EVALUATION_QUERIES
] + [
    {
        "user_input": "What is the total climate change GWP of the IPG F4090-05 machine roll tape per square meter?",
        "reference": "2.03E-01 kg CO2 eq. per m2: 1.29E-01 upstream, 6.69E-02 core and 7.50E-03 downstream.",
        "category": "direct_fact",
        "sources": ["EPD_tape_IPG_hotmelt.pdf"],
    },
    {
        "user_input": "What is the weight and static load capacity of the StabilPlastik EP 08 pallet?",
        "reference": "It weighs 25 kg and has a static load capacity of 10,000 kg.",
        "category": "direct_fact",
        "sources": ["EPD_pallet_stabilplastik_ep08.pdf"],
    },
    {
        "user_input": "How do the EU climate reporting guidelines define transition risks?",
        "reference": "Risks to the company that arise from the transition to a low-carbon and climate-resilient economy, such as policy, legal, technology, market and reputational risks.",
        "category": "direct_fact",
        "sources": ["REF_eu_climate_reporting_guidelines.pdf"],
    },
    {
        "user_input": "What is the Logypal 1 made of according to its 2021 datasheet?",
        "reference": "100% post-consumer recycled plastic, mainly from end-of-life agricultural packaging (silage film) and industrial packaging waste.",
        "category": "direct_fact",
        "sources": ["ART_relicyc_logypal1_datasheet_2021.pdf"],
    },
]
print(f"{len(manual_samples)} hand-written samples")

## 2. Synthetic Samples

Sample chunks from the vector store, skipping very short ones (titles, page footers), and ask the LLM for one question and answer grounded only in each chunk.

In [ ]:
SAMPLE_SIZE = 10

random.seed(42)
records = await vector_store.get_chunks_by_filter()
candidates = [r for r in records if len(r.content) > 300]
sampled = random.sample(candidates, min(SAMPLE_SIZE, len(candidates)))
print(f"Sampled {len(sampled)} of {len(candidates)} chunks")

In [ ]:
GENERATOR_PROMPT = """Write one question and answer based strictly on the context below.

- Ask what a procurement officer or sustainability auditor would ask.
- The answer must be factual, concise and derived only from the context.
- Mention specific products, IDs or dates from the context.

Reply exactly in this format:
QUESTION: <question>
ANSWER: <answer>

Context:
{context}"""

synthetic_samples = []
for record in sampled:
    prompt = GENERATOR_PROMPT.format(context=record.content)
    response = await llm.generate(
        [
            LLMMessage(
                role=Roles.USER, content=[MessageContent(type="text", text=prompt)]
            )
        ]
    )
    text = response.content[0].text or ""
    if "ANSWER:" not in text:
        print(f"Skipped unparsable response for {record.metadata['source_file']}")
        continue
    question, answer = text.split("ANSWER:", 1)
    synthetic_samples.append(
        {
            "user_input": question.replace("QUESTION:", "").strip(),
            "reference": answer.strip(),
            "category": "synthetic",
            "sources": [record.metadata["source_file"]],
        }
    )

for sample in synthetic_samples[:3]:
    print(
        f"[{sample['sources'][0]}]\nQ: {sample['user_input']}\nA: {sample['reference']}\n"
    )

Synthetic questions mirror the chunk's wording, which makes retrieval easier than real user questions, and the LLM can misread the chunk. Review a sample before relying on it.

## 3. Export

In [ ]:
all_samples = manual_samples + synthetic_samples
output_path = DB_DIR / "evaluation_dataset.json"
output_path.write_text(json.dumps(all_samples, indent=2, ensure_ascii=False))
print(f"Saved {len(all_samples)} samples to {output_path}")

To evaluate with it, use `user_input` and `reference` in place of `query` and `ground_truth_answer` in T04.